# 04 Null Analysis

Computes null percentages using chunking/row groups and checks critical columns.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "04_null_analysis"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 04_null_analysis
Start time: 2026-06-01 17:59:46.875720
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [2]:
inventory_path = ROOT / "eda" / "bronze" / "outputs" / "tables" / "01_file_integrity" / "file_integrity.csv"
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
else:
    inventory = build_file_inventory(RAW_DATA_PATH, EXPECTED_ENDPOINTS)

endpoint_column_cache = {}
endpoint_row_cache = {}
for endpoint in EXPECTED_ENDPOINTS:
    observed = []
    endpoint_rows = inventory[inventory["endpoint"].eq(endpoint)]
    endpoint_row_cache[endpoint] = int(endpoint_rows["row_count"].sum()) if "row_count" in endpoint_rows else 0
    for value in endpoint_rows["columns"].dropna().astype(str):
        for column in value.split("|"):
            if column and column not in observed:
                observed.append(column)
    endpoint_column_cache[endpoint] = observed

def classify_null(endpoint: str, column: str, null_count: int) -> tuple[str, str]:
    if null_count == 0:
        return "PASS", "NO_NULLS"
    if f"{endpoint}.{column}" in ALLOWED_NULL_SCENARIOS:
        return "PASS", "ALLOWED_STRUCTURAL_NULL"
    if column in TECHNICAL_KEY_COLUMNS:
        return "FAIL", "TECHNICAL_KEY_NULL"
    if column in DOMAIN_REVIEW_COLUMNS:
        return "REVIEW", "DOMAIN_SEMANTIC_NULL"
    if column in STRUCTURAL_OPTIONAL_COLUMNS:
        return "PASS", "STRUCTURAL_OR_OPTIONAL_NULL"
    if column in CRITICAL_COLUMNS.get(endpoint, []):
        return "REVIEW", "CONFIG_CRITICAL_REVIEW"
    return "PASS", "OPTIONAL_SOURCE_NULL"

profiles = []
chunk_size = int(THRESHOLDS.get("validation_chunk_size", 200000))
for endpoint in EXPECTED_ENDPOINTS:
    expected_columns = endpoint_column_cache.get(endpoint, [])
    is_telemetry = endpoint in TELEMETRY_ENDPOINTS
    profile = null_profile(
        RAW_DATA_PATH,
        endpoint,
        is_telemetry=is_telemetry,
        chunksize=chunk_size,
        expected_columns=expected_columns,
    )
    if profile.empty and expected_columns:
        profile = pd.DataFrame([
            {"endpoint": endpoint, "column": column, "rows": endpoint_row_cache.get(endpoint, 0), "null_count": 0, "null_pct": 0.0}
            for column in expected_columns
        ])
    if profile.empty:
        continue
    profile["critical"] = profile["column"].map(lambda column: column in CRITICAL_COLUMNS.get(endpoint, []))
    classified = profile.apply(lambda row: classify_null(row["endpoint"], row["column"], int(row["null_count"])), axis=1)
    profile["status"] = [item[0] for item in classified]
    profile["null_class"] = [item[1] for item in classified]
    profiles.append(profile)

null_df = pd.concat(profiles, ignore_index=True) if profiles else pd.DataFrame()
null_df = null_df.sort_values(["status", "null_pct", "endpoint", "column"], ascending=[True, False, True, True])
null_df.to_csv(OUTPUT_TABLES / "null_percentage.csv", index=False)
display(null_df.sort_values("null_pct", ascending=False).head(80))

,endpoint,column,rows,null_count,null_pct,critical,status,null_class
115,pit,stop_duration,7871,6904,87.714395,False,PASS,OPTIONAL_SOURCE_NULL
123,race_control,driver_number,9736,8173,83.946179,False,PASS,ALLOWED_STRUCTURAL_NULL
129,race_control,qualifying_phase,9736,7747,79.570666,False,PASS,OPTIONAL_SOURCE_NULL
128,race_control,sector,9736,7388,75.883320,False,PASS,OPTIONAL_SOURCE_NULL
44,drivers,country_code,2837,1639,57.772295,False,PASS,ALLOWED_STRUCTURAL_NULL
...,...,...,...,...,...,...,...,...
161,location,driver_number,32887867,0,0.000000,True,PASS,NO_NULLS
163,location,lap_number,32887867,0,0.000000,True,PASS,NO_NULLS
162,location,session_key,32887867,0,0.000000,True,PASS,NO_NULLS
12,meetings,circuit_image,76,0,0.000000,False,PASS,NO_NULLS


In [3]:
plot_df = null_df[null_df["null_count"] > 0].sort_values("null_pct", ascending=False).head(50)
fig = px.bar(plot_df, x="null_pct", y="endpoint", color="column", orientation="h", title="Top Null Percentages by Endpoint Column")
fig.write_html(OUTPUT_CHARTS / "null_percentage.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "null_percentage.png")
except Exception:
    pass
fig.show()

In [4]:
failed = null_df[null_df["status"] == "FAIL"] if not null_df.empty else pd.DataFrame()
review = null_df[null_df["status"] == "REVIEW"] if not null_df.empty else pd.DataFrame()
report = {"notebook": NOTEBOOK_NAME, "timestamp": datetime.now().isoformat(), "p0_status": "PASS" if failed.empty else "FAIL", "review_count": int(len(review)), "results": null_df.to_dict("records") if not null_df.empty else []}
write_report("null_analysis", report)
write_insight(
    "04_null_insights.md",
    "Null Analysis Insights",
    f"Computed null profile for {null_df['endpoint'].nunique() if not null_df.empty else 0} endpoints and {len(null_df)} endpoint-columns.",
    [f"Columns with nulls: {int((null_df['null_count'] > 0).sum()) if not null_df.empty else 0}", f"Review-level semantic nulls: {len(review)}"],
    [f"{row.endpoint}.{row.column}: {row.null_count} nulls classified as {row.null_class}" for row in failed.itertuples()],
    ["Treat technical key nulls as P0 only.", "Document domain semantic nulls as REVIEW for Silver strategy.", "Keep structural optional nulls out of P0 gating."],
    ["Run 05_range_validation.ipynb"],
)
print(report["p0_status"])

PASS
